# Signatures comparison — new sorted-cell test cohort (CHESS-1333 / OD-128)

Companion to the published Figure 4 notebook (`Signatures comparison.ipynb`). The new sorted-cell cohort delivered with [Jira OD-128](https://bostongene.atlassian.net/browse/OD-128) serves as a true held-out **test** cohort, satisfying Reviewer 1's request to "Add new data of sorted cells to cell signature comparison" and the paper Methods commitment to ~75/25 train/test separation.

**Scope:** 16 of 20 FGES. The four rare-GOI FGES — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — are deferred to a separate rare-types notebook that uses 75/25 stratified holdouts on the original cohort (helpers ship in `signature_validation.benchmark.splits`).

**Random-FGES baseline:** v1 random gene lists are reused (loaded from `msigdb_gmt.pkl`) but rescored on the new cohort so ranks stay comparable.

**Where things land:** pickle and SVGs go to `/home/jovyan/SignValArticle/...` with a `_new_cohort` suffix; v1 outputs are not overwritten.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
    load_new_cohort_expressions,
)
from signature_validation.benchmark.plotting import (
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import (
    compute_mapping_ssgseas,
    compute_out_table,
    fdr_correct_out,
)
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import cells_p

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

In [3]:
NEW_ANNOT_PATH = Path(
    "/home/jovyan/projects/SignVal/Signature_validation/sorted_cells_to_check_all_annot.tsv"
)

OUTPUT_DIR = Path("/home/jovyan/SignValArticle/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_SSGSEAS_PATH = OUTPUT_DIR / "mapping_ssgseas_new_cohort.pkl"
FGES_METRICS_PATH = OUTPUT_DIR / "fges_metrics_new_cohort.pkl"
OUT_TSV_PATH = OUTPUT_DIR / "out_new_cohort.tsv"
HEATMAP_PATH = OUTPUT_DIR / "signature_heatmap_new_cohort.svg"

# ── Режим работы ──────────────────────────────────────────────────────────
# RECOMPUTE=True  — пересчитать ssGSEA (mapping_ssgseas) и fges_metrics с нуля
#                   (медленно; нужны экспрессии с S3), затем сохранить pickle.
# RECOMPUTE=False — загрузить готовые pickle (MAPPING_SSGSEAS_PATH /
#                   FGES_METRICS_PATH) и только строить графики/таблицы
#                   (экспрессии не грузятся).
RECOMPUTE = True

logger.info("RECOMPUTE={}", RECOMPUTE)
logger.info("new annotation:    {}", NEW_ANNOT_PATH)
logger.info("mapping_ssgseas:   {}", MAPPING_SSGSEAS_PATH)

2026-07-28 14:42:19.045 | INFO     | __main__:<module>:11 - new annotation:    /home/jovyan/projects/SignVal/Signature_validation/sorted_cells_to_check_all_annot.tsv
2026-07-28 14:42:19.045 | INFO     | __main__:<module>:12 - mapping_ssgseas:   /home/jovyan/SignValArticle/mapping_ssgseas_new_cohort.pkl


In [4]:
public_cells_annot = load_new_cohort_annotation(NEW_ANNOT_PATH)
public_cells_annot["Cell_type"].value_counts()

/home/jovyan/projects/Signature_Validation/src/signature_validation/utils/utils.py:541: DtypeWarning: Columns (4,28,30,31,33,34,35,36,39,41,44,47,52,53,54,55,57,59) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(
2026-07-12 22:43:33.374 | INFO     | signature_validation.benchmark.cohorts:load_new_cohort_annotation:226 - loaded 30418 samples across 217 cell types from /home/jovyan/projects/SignVal/Signature_validation/sorted_cells_to_check_all_annot.tsv


Cell_type
Epithelium              3155
Macrophages             2404
CD4_T_cells             2379
Monocytes               2248
CD8_T_cells             2009
                        ... 
Mature_neutrophils         1
Trophoblast_cells          1
Cytotoxic_NK_cells         1
Intestinal_organoids       1
Schwann_cells              1
Name: count, Length: 217, dtype: int64

In [5]:
from signature_validation.utils.utils import read_expressions

In [6]:
# Экспрессии нужны только для пересчёта (ssGSEA + метрики).
if RECOMPUTE:
    EXPR_S3_PATH = "/uftp2/Databases/Deconvolution/"
    public_cells_expr = read_expressions(public_cells_annot, path=EXPR_S3_PATH)
    logger.info("expressions: {}", public_cells_expr.shape)
else:
    public_cells_expr = None
    logger.info("RECOMPUTE=False — загрузку экспрессий пропускаю")

no GSE58310 expression
no GSE101993 expression
no GSE141217 expression
no GSE152446 expression
no GSE154122 expression
no GSE177862 expression
no GSE184398 expression
no GSE191279 expression
no GSE193682 expression
no GSE201152 expression
no GSE77312 expression
no GSE83492 expression
no GSE84135 expression
no PRJNA360082 expression
no PRJNA374973 expression
no PRJNA397967 expression
no PRJNA434217 expression
no PRJNA436739 expression
no PRJNA484735 expression
no PRJNA491656 expression
no PRJNA596741 expression
no PRJNA624366 expression
no PRJNA631458 expression
no GSE223806 expression
no GSE171256 expression
no GSE152590 expression
no GSE213696 expression
no GSE173387 expression
no GSE221563 expression
no GSE235755 expression
no GSE162712 expression
no GSE199490 expression
no GSE226249 expression
no GSE247226 expression
no GSE188464 expression
no GSE173635 expression
no GSE184307 expression
no PRJNA562324 expression
no GSE172372 expression
no GSE215144 expression
no GSE217012 expressio

(20062, 23440)

In [ ]:
V1_GMT_PICKLE = "/home/jovyan/projects/SignVal/Signature_validation/article/msigdb_gmt.pkl"

v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)

for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(v1_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"

if RECOMPUTE:
    msigdb_gmt = harmonize_gmt_to_index(v1_gmt, public_cells_expr.index)
else:
    # Без пересчёта нужны только ИМЕНА суб-сигнатур (не гены) — берём v1_gmt как есть.
    msigdb_gmt = v1_gmt
logger.info(
    "msigdb_gmt: {} FGES, {} signatures total",
    len(msigdb_gmt),
    sum(len(v) for v in msigdb_gmt.values()),
)

ModuleNotFoundError: No module named 'bioreactor'

In [ ]:
mapping = build_mapping(annotation=public_cells_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, public_cells_annot)
logger.info(
    "in-scope: {} FGES; controls present in new cohort: {}",
    len(mapping),
    len(controls_present),
)
for sign, bucket in mapping.items():
    logger.info(
        "{}: GOI={}, Control={}, Deleted={}",
        sign,
        bucket["Goi"],
        len(bucket["Control"]),
        len(bucket["Deleted_controls"]),
    )

In [ ]:
from signature_validation.benchmark.cohorts import PARENT_TO_DAUGHTER
from signature_validation.benchmark.scoring import clean_parent_daughter_goi

if RECOMPUTE:
    mapping_ssgseas = compute_mapping_ssgseas(
        public_cells_expr=public_cells_expr,
        public_cells_annot=public_cells_annot,
        mapping=mapping,
        msigdb_gmt=msigdb_gmt,
    )
    # Очистка parent→daughter (Scater_plots ячейка 16): без неё скоры
    # Macrophages/Monocyte искажены (общие сигнатуры остаются в GOI родителя).
    mapping_ssgseas = clean_parent_daughter_goi(mapping_ssgseas, PARENT_TO_DAUGHTER)
    with open(MAPPING_SSGSEAS_PATH, "wb") as fh:
        pickle.dump(mapping_ssgseas, fh, pickle.HIGHEST_PROTOCOL)
    logger.info("computed+cleaned mapping_ssgseas → {}", MAPPING_SSGSEAS_PATH)
else:
    with open(MAPPING_SSGSEAS_PATH, "rb") as fh:
        mapping_ssgseas = pickle.load(fh)
    logger.info("loaded mapping_ssgseas from {}", MAPPING_SSGSEAS_PATH)

for sign in mapping_ssgseas:
    if sign in EXCLUDED_FGES_RARE:
        continue
    assert mapping_ssgseas[sign]["Goi"], f"{sign}: GOI cohort is empty"

## Score correctness, metrics, scatters and Supplement tables

`clean_parent_daughter_goi` reproduces the v1 `parent_to_daughter` cleaning (Scater_plots cell 16) that the first draft omitted — without it macrophage/monocyte GOI frames keep daughter-shared signatures and the heatmap / `out` table / scatter are wrong. Then `fges_metrics` (bootstrap F1/AUC + rank CV), the F1-vs-CV source scatters (Fig 4 F/G) and the S4/S6 Supplement tables.

In [ ]:
# Проверка счётчиков образцов для макрофагов — только в режиме пересчёта
# (нужны экспрессии): аннотация vs. экспрессии vs. GOI-фреймы.
if RECOMPUTE:
    MACRO_GOI = {
        "Main4_Pan_macrophage_signature": "Macrophages",
        "Main4_M2_signature": "Macrophages_M2",
        "Main4_Monocyte": "Monocytes",
    }

    expr_cols = set(public_cells_expr.columns)
    ct_counts = public_cells_annot["Cell_type"].value_counts()

    for fges, ct in MACRO_GOI.items():
        n_annot = int(ct_counts.get(ct, 0))
        samples_of_ct = public_cells_annot.index[public_cells_annot["Cell_type"] == ct]
        n_expr = len(expr_cols.intersection(samples_of_ct))
        goi_frames = mapping_ssgseas.get(fges, {}).get("Goi", {})
        n_goi = int(goi_frames[ct].shape[0]) if ct in goi_frames else 0
        dropped = n_annot - n_expr
        logger.info(
            "{ct:16s} | FGES={fges:34s} | annot={n_annot:5d} | with_expr={n_expr:5d} "
            "| GOI_frame_rows={n_goi:5d} | dropped_no_expr={dropped:5d}",
            ct=ct, fges=fges, n_annot=n_annot, n_expr=n_expr, n_goi=n_goi, dropped=dropped,
        )
        if n_goi != n_expr:
            logger.warning(
                "{ct}: GOI-фрейм ({n_goi}) != образцов с экспрессиями ({n_expr}) — "
                "проверь дубликаты индексов / фильтрацию",
                ct=ct, n_goi=n_goi, n_expr=n_expr,
            )

    all_cts = {ct for b in mapping.values() for g in ("Goi", "Control", "Deleted_controls") for ct in b[g]}
    total_samples = public_cells_annot.index[public_cells_annot["Cell_type"].isin(all_cts)]
    total_with_expr = len(expr_cols.intersection(total_samples))
    logger.info("ОБЩЕЕ образцов (все in-scope cell types, с экспрессиями): {}", total_with_expr)
else:
    logger.info("RECOMPUTE=False — проверку счётчиков макрофагов пропускаю")

In [ ]:
out = compute_out_table(mapping_ssgseas, mapping, msigdb_gmt, controls_present)
out = fdr_correct_out(out, controls_present)
out.to_csv(OUT_TSV_PATH, sep="\t")
logger.info("wrote {} ({} rows × {} cols)", OUT_TSV_PATH, *out.shape)
out.head()

In [ ]:
plot_violin_per_source(mapping_ssgseas, save_dir=OUTPUT_DIR)
plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas,
    out_df=out,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
    annotation=public_cells_annot,
    controls_order=controls_present,
    palette={ct: cells_p[ct] for ct in controls_present if ct in cells_p},
    save_path=HEATMAP_PATH,
    short=True,
)
averaged = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
)
logger.info("plots saved under {}", OUTPUT_DIR)

In [ ]:
# fges_metrics: bootstrap-классификация (F1/Accuracy/PR-AUC/ROC-AUC) + CV рангов
# (порт Scater_plots ячейки 20). Пересчёт или загрузка готового pickle.
from signature_validation.benchmark.metrics import compute_fges_metrics

if RECOMPUTE:
    ranked_expr = public_cells_expr.rank(pct=True)
    pipeline_genes = public_cells_expr.index.to_list()
    # Дроп Th2_cells только для метрик (Scater_plots ячейка 19): get_strat_cell_type
    # берёт min_samples по контролям, крошечный Th2 обрезал бы все подвыборки.
    annot_for_metrics = public_cells_annot[public_cells_annot["Cell_type"] != "Th2_cells"]
    fges_metrics = compute_fges_metrics(
        mapping_ssgseas=mapping_ssgseas,
        msigdb_gmt=msigdb_gmt,
        public_cells_annot=annot_for_metrics,
        ranked_expr=ranked_expr,
        pipeline_genes=pipeline_genes,
        n_iter=10,
    )
    with open(FGES_METRICS_PATH, "wb") as fh:
        pickle.dump(fges_metrics, fh, pickle.HIGHEST_PROTOCOL)
    logger.info("computed fges_metrics → {} ({} FGES cols)", FGES_METRICS_PATH, len(fges_metrics))
else:
    with open(FGES_METRICS_PATH, "rb") as fh:
        fges_metrics = pickle.load(fh)
    logger.info("loaded fges_metrics from {} ({} FGES cols)", FGES_METRICS_PATH, len(fges_metrics))

In [ ]:
# Скаттеры F1 vs CV рангов, по источникам FGES (как Scater_plots панели F/G).
# Пишем в OUTPUT_DIR (hub scratch), чтобы НЕ перезаписать опубликованные paper-SVG.
from signature_validation.benchmark.metrics import plot_f1_cv_scatters

plot_f1_cv_scatters(
    fges_metrics=fges_metrics,
    msigdb_gmt=msigdb_gmt,
    save_dir=OUTPUT_DIR,
)
logger.info("F1/CV scatters → {}", OUTPUT_DIR / "svg_pictures_F1_cv")

In [ ]:
# Supplementary таблицы: S4.x (перформанс FGES на клетку, как пример B_cells)
# + S6.1 (список датасетов из аннотации).
import os

from signature_validation.benchmark.tables import (
    build_dataset_list_table,
    build_fges_performance_tables,
)

TABLES_DIR = (OUTPUT_DIR / "tables").resolve()
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# detect_fges_source читает ./data/msigdb...gmt относительно cwd — временно
# переходим на уровень выше (Cell_type_FGES_comparison), где эта папка есть.
_cwd0 = os.getcwd()
try:
    os.chdir("..")
    s4_tables = build_fges_performance_tables(
        mapping_ssgseas=mapping_ssgseas,
        fges_metrics=fges_metrics,
        mapping=mapping,
        msigdb_gmt=msigdb_gmt,
        save_dir=TABLES_DIR,
        prefix="S4",
    )
finally:
    os.chdir(_cwd0)

s6_table = build_dataset_list_table(
    public_cells_annot,
    TABLES_DIR / "S6.1_sorted_cell_datasets.tsv",
)
logger.info(
    "wrote {} S4 tables + S6.1 ({} datasets) → {}",
    len(s4_tables), len(s6_table), TABLES_DIR,
)
s6_table.head()

## Rare cell types (out of scope here)

FGES whose GOI is rare in the new cohort — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — are deferred to a separate notebook authored by Nadezhda. That notebook reuses the original cohort (`/uftp2/.../cells_all_annotation.tsv` plus the Tonsillar Tfh / Mast / Endothelium_lymph patches), generates 10 stratified 75/25 holdouts via `signature_validation.benchmark.splits.stratified_holdout_indices` (stratified by BG-FGES score median × GOI/Control), scores ssGSEA on each test fold and aggregates with `aggregate_score_over_splits`. Rare cell types are starred on the resulting figures.